In [ ]:
# CELL 1 — Runtime information (GPU is optional for this cell)
import importlib.metadata as md, os, platform, shutil, sys
from pathlib import Path
print('Python:', sys.version)
print('OS:', platform.platform())
try:
    import torch
    print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except ImportError:
    print('torch: not installed')
for package in ('scikit-learn', 'transformers'):
    try: print(package + ':', md.version(package))
    except md.PackageNotFoundError: print(package + ': not installed')
print('Disk free GiB:', round(shutil.disk_usage('/content' if Path('/content').exists() else '.').free / 1024**3, 2))

In [ ]:
# CELL 2 — GitHub configuration. Set a full commit SHA; no branch/latest fallback.
REPO_URL = ''  # e.g. https://github.com/ORG/REPO.git
COMMIT_SHA = ''  # required: full 40-character commit SHA
WORKDIR = '/content/absa-project'
DATA_DIR = ''  # optional separate directory containing the four frozen CSV files
MODEL_ASSETS_DIR = ''  # optional local/Drive path to exact fingerprinted PhoBERT tokenizer and VnCoreNLP files
GOOGLE_DRIVE_ARTIFACT_DIR = ''  # optional; configured in CELL 7
MOUNT_GOOGLE_DRIVE = False
import re
if not REPO_URL.strip(): raise ValueError('Set REPO_URL before cloning')
if not re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA): raise ValueError('Set COMMIT_SHA to a full 40-character SHA')

In [ ]:
# CELL 3 — Clone and checkout exactly COMMIT_SHA; stop on mismatch or dirty state.
import subprocess
repo_dir = Path(WORKDIR)
if repo_dir.exists():
    if not (repo_dir / '.git').exists(): raise RuntimeError(f'WORKDIR exists but is not a git checkout: {repo_dir}')
else:
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(repo_dir)], check=True)
subprocess.run(['git', 'fetch', '--no-tags', 'origin', COMMIT_SHA], cwd=repo_dir, check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT_SHA], cwd=repo_dir, check=True)
head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo_dir, check=True, capture_output=True, text=True).stdout.strip()
if head.lower() != COMMIT_SHA.lower(): raise RuntimeError(f'HEAD {head} does not match requested commit')
dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=repo_dir, check=True, capture_output=True, text=True).stdout
if dirty.strip(): raise RuntimeError('Checkout is dirty; inspect before continuing:\n' + dirty)
print('Verified clean detached checkout:', head)

In [ ]:
# CELL 4 — Dataset location and byte hashes. This cell never cleans or rewrites data.
import hashlib, json
repo_data = repo_dir / 'data'
data_dir = Path(DATA_DIR).expanduser() if DATA_DIR else repo_data
if DATA_DIR and not data_dir.exists() and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
protocol = json.loads((repo_dir / 'protocols/evaluation_protocol_v1.yaml').read_text(encoding='utf-8'))
for split, meta in protocol['source_files'].items():
    source = data_dir / Path(meta['path']).name
    if not source.is_file(): raise FileNotFoundError(f'Missing frozen {split} CSV: {source}. If staged on Drive, mount in CELL 7, set DATA_DIR, then rerun this cell.')
    actual = hashlib.sha256(source.read_bytes()).hexdigest()
    if actual != meta['sha256']: raise ValueError(f'Frozen CSV SHA-256 mismatch: {source}')
    print(split, meta['rows'], actual)
print('Dataset bytes verified; use only official train/validation for fitting and selection.')

In [ ]:
# CELL 5 — Environment diagnostics and explicit install instructions (not executed).
import importlib.metadata as md, unicodedata
registration = json.loads((repo_dir / 'protocols/experiments/experiment_registration_v1.json').read_text())
expected_runtime = registration['runtime']['modeling']
expected_versions = {'torch': '2.9.0', 'scikit-learn': '1.7.2', 'transformers': '4.57.1'}
problems = []
if platform.python_version() != expected_runtime['python']: problems.append(f"Python {platform.python_version()} != registered {expected_runtime['python']}")
if unicodedata.unidata_version != expected_runtime['unicode']: problems.append(f"Unicode {unicodedata.unidata_version} != registered modeling UCD {expected_runtime['unicode']}")
for package, version in expected_versions.items():
    try: actual = md.version(package)
    except md.PackageNotFoundError: actual = 'missing'
    print(package, 'installed=', actual, 'registered=', version)
    if actual != version: problems.append(f'{package} {actual} != {version}')
lock = repo_dir / 'protocols/experiments/modeling_requirements_v1.lock'
tokenizer_pins = repo_dir / 'reports/modeling_preflight/tokenizer_environment_v1.txt'
print('Registered torch/sklearn install command (not executed): python -m pip install --require-hashes -r', lock)
print('Pinned tokenizer dependency command (not executed): python -m pip install -r', tokenizer_pins)
if problems: raise RuntimeError('Registered modeling environment is incompatible: ' + '; '.join(problems))
print('Modeling runtime matches registration. This is distinct from the Python 3.15.1 / Unicode 17 data gate.')

In [ ]:
# CELL 6 — Portable byte integrity only; does not rerun exact-runtime normalization.
portable_cmd = [sys.executable, str(repo_dir / 'src/portable_verification.py'),
                '--root', str(repo_dir), '--commit-sha', COMMIT_SHA]
if MODEL_ASSETS_DIR: portable_cmd += ['--assets-dir', str(Path(MODEL_ASSETS_DIR).expanduser())]
portable = subprocess.run(portable_cmd,
                          cwd=repo_dir, check=True, capture_output=True, text=True)
portable_result = json.loads(portable.stdout)
assert portable_result['exact_runtime_reproduced'] is False
print(json.dumps(portable_result, indent=2))
print('Historical local exact-runtime receipt:', portable_result['gate_runtime']['historical_receipt'])

In [ ]:
# CELL 7 — Optional Google Drive mount for durable artifacts or separately staged private data.
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('Google Drive mounting is available only in Colab') from exc
    if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
artifact_dir = Path(GOOGLE_DRIVE_ARTIFACT_DIR).expanduser() if GOOGLE_DRIVE_ARTIFACT_DIR else None
if artifact_dir:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    print('Durable artifact backup directory:', artifact_dir)
    print('Use /content local disk for active data and temporary checkpoints; copy finalized artifacts to Drive explicitly.')
else:
    print('Drive not mounted. Set MOUNT_GOOGLE_DRIVE=True and GOOGLE_DRIVE_ARTIFACT_DIR to opt in.')

In [ ]:
# CELL 8 — CPU synthetic smoke only; generated artificial rows, no official CSV/model weights.
smoke = subprocess.run([sys.executable, '-m', 'src.training.synthetic_smoke', '--cpu-only'],
                       cwd=repo_dir, check=True, capture_output=True, text=True)
print(smoke.stdout)
if smoke.stderr: print(smoke.stderr)

In [ ]:
# CELL 9 — Future launch gate status. This cell only displays registration; it never trains.
for family, config in registration['families'].items():
    print(family, [candidate['id'] for candidate in config['candidates']])
print('Seeds:', registration['seeds'], '| maximum registered fits:', registration['budget']['maximum_main_training_runs'])
print('Neural target: single CUDA GPU >=16 GiB; verify driver/CUDA and deterministic behavior at launch.')
print('Pending: TRAIN-02B GPU smoke; TRAIN-03 run manifest, checkpoint/resume and authorized guarded launch.')
print('TRAIN-04 registered experiments have not started. No training command is run from this notebook.')